# Preprocessing

In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler, LabelEncoder

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("../data/dataset.csv", index_col= "Unnamed: 0") # load dataset

df.head() # display top rows

,open,high,low,close,volume,marketCap,timestamp,crypto_name,date
0,112.900002,118.800003,107.142998,115.910004,0.0,1.288693e+09,2013-05-05T23:59:59.999Z,Bitcoin,2013-05-05
1,3.493130,3.692460,3.346060,3.590890,0.0,6.229819e+07,2013-05-05T23:59:59.999Z,Litecoin,2013-05-05
2,115.980003,124.663002,106.639999,112.300003,0.0,1.249023e+09,2013-05-06T23:59:59.999Z,Bitcoin,2013-05-06
3,3.594220,3.781020,3.116020,3.371250,0.0,5.859436e+07,2013-05-06T23:59:59.999Z,Litecoin,2013-05-06
4,112.250000,113.444000,97.699997,111.500000,0.0,1.240594e+09,2013-05-07T23:59:59.999Z,Bitcoin,2013-05-07


In [3]:
print(df.shape) # print shape of df
print(df.columns) # print columns of df
df.info()

(72946, 9)
Index(['open', 'high', 'low', 'close', 'volume', 'marketCap', 'timestamp',
       'crypto_name', 'date'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 72946 entries, 0 to 72945
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   open         72946 non-null  float64
 1   high         72946 non-null  float64
 2   low          72946 non-null  float64
 3   close        72946 non-null  float64
 4   volume       72946 non-null  float64
 5   marketCap    72946 non-null  float64
 6   timestamp    72946 non-null  str    
 7   crypto_name  72946 non-null  str    
 8   date         72946 non-null  str    
dtypes: float64(6), str(3)
memory usage: 5.0 MB


In [4]:
# Converting to datetime format
df['date'] = pd.to_datetime(df['date'])

df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', errors='coerce')

In [5]:
# Sorting is necessary for creating volatility column
df = df.sort_values(['crypto_name','date']).reset_index(drop=True)

In [6]:
# Checking for null values
df.isnull().sum()

open           0
high           0
low            0
close          0
volume         0
marketCap      0
timestamp      0
crypto_name    0
date           0
dtype: int64

In [7]:
df.drop(columns=['timestamp'], inplace=True) # We have date column and time has no variance

In [8]:
df = df.drop_duplicates() # Dropping duplicates

# Feature Engineering

In [9]:
# Price Range
df['price_range'] = df['high'] - df['low']

In [10]:
# Daily Return
df['return'] = (df['close'] - df['open']) / df['open']

In [11]:
# Log Return
df['log_return'] = np.log(df['close'] / df['close'].shift(1))

In [12]:
# Volatility Proxy
df['volatility'] = (df['high'] - df['low']) / df['open']

In [13]:
# Time Features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek

In [14]:
# Rolling Features (Very Useful)
df['rolling_mean_7'] = df['close'].rolling(7).mean()
df['rolling_std_7'] = df['close'].rolling(7).std()

df['rolling_mean_14'] = df['close'].rolling(14).mean()
df['rolling_std_14'] = df['close'].rolling(14).std()

In [15]:
# Volume Features
df['volume_change'] = df['volume'].pct_change()

In [16]:
# MarketCap Features
df['marketcap_change'] = df['marketCap'].pct_change()

In [17]:
# Encode Crypto Name
le = LabelEncoder()

df['crypto_id'] = le.fit_transform(df['crypto_name'])

joblib.dump(le, "../models/encoder.pkl")

['../models/encoder.pkl']

In [19]:
# Removing NaN Created by Rolling
df = df.dropna().reset_index(drop=True)

In [20]:
# Replace infinite values
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Check missing values
print(df.isnull().sum())

# Fill missing values
df = df.ffill()
df = df.bfill()

# If still present
df = df.dropna().reset_index(drop=True)

open                 0
high                 0
low                  0
close                0
volume               0
marketCap            0
crypto_name          0
date                 0
price_range          0
return               1
log_return           0
volatility           1
year                 0
month                0
day                  0
dayofweek            0
rolling_mean_7       0
rolling_std_7        0
rolling_mean_14      0
rolling_std_14       0
volume_change        5
marketcap_change    36
crypto_id            0
dtype: int64


In [21]:
# Feature Scaling
scaler = StandardScaler()

scale_cols = [
'open','high','low','close',
'volume','marketCap',
'price_range','return','log_return',
'rolling_mean_7','rolling_std_7',
'rolling_mean_14','rolling_std_14',
'volume_change','marketcap_change'
]

df[scale_cols] = scaler.fit_transform(df[scale_cols])

joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']

In [48]:
# Useful features in crypto
df['coin_return_mean_7'] = df.groupby('crypto_id')['return'].rolling(7).mean().reset_index(0,drop=True)

df['coin_return_std_7'] = df.groupby('crypto_id')['return'].rolling(7).std().reset_index(0,drop=True)

df['coin_volume_mean_7'] = df.groupby('crypto_id')['volume'].rolling(7).mean().reset_index(0,drop=True)

In [49]:
df.drop(columns=["crypto_name", "date"], inplace=True)

In [50]:
df = df.dropna().reset_index(drop=True)

In [51]:
print(df.isna().sum())

open                  0
high                  0
low                   0
close                 0
volume                0
marketCap             0
price_range           0
return                0
log_return            0
volatility            0
year                  0
month                 0
day                   0
dayofweek             0
rolling_mean_7        0
rolling_std_7         0
rolling_mean_14       0
rolling_std_14        0
volume_change         0
marketcap_change      0
crypto_id             0
coin_return_mean_7    0
coin_return_std_7     0
coin_volume_mean_7    0
dtype: int64


In [52]:
df.shape

(69542, 24)

In [53]:
df.to_csv("../data/crypto_processed.csv", index=False)